In [ ]:
import pandas as pd
import numpy as np
# import os
# os.environ["XLA_FLAGS"] = "--xla_cpu_multi_thread_eigen=true --xla_cpu_multi_thread_eigen_num_threads=4"
import numpyro
numpyro.set_host_device_count(4)
import jax
import jax.numpy as jnp
from jax import lax
from jax import vmap
from numpyro import distributions as dist
from numpyro.infer import MCMC, NUTS, init_to_median
import matplotlib.pyplot as plt

df = pd.read_csv('source/df.csv')
df

In [ ]:
response = df["sales"].values
# response_norm = response.mean()
response_norm = 1.
y = np.log(response / response_norm)
y = jnp.array(y)

In [ ]:
def make_fourier_series(t, period, order):
    """
    Args
    ----
    t: array-like, time points at which to evaluate the Fourier series
    period: int, the period of the seasonality
    order: int, the number of Fourier terms to include
    """
    sin_terms = jnp.array([jnp.sin(2 * jnp.pi * i * t / period) for i in range(1, order + 1)])
    cos_terms = jnp.array([jnp.cos(2 * jnp.pi * i * t / period) for i in range(1, order + 1)])
    return jnp.concatenate((sin_terms, cos_terms), axis=0).transpose(1, 0)

x_glb_trend = jnp.arange(len(y)) / 365.25
x_annual_seas = make_fourier_series(jnp.arange(len(y)), period=365.25, order=3)
x_weekly_seas = make_fourier_series(jnp.arange(len(y)), period=7, order=2)
x_seas = jnp.concatenate((x_annual_seas, x_weekly_seas), axis=1)
print(x_seas.shape)

In [ ]:
def dlt_transition_step(carry, inputs, lev_sm, slp_sm, theta):
    """Damped Local Trend (DLT) transition function
    Args
    ----
    carry: tuple containing the previous level and bias
    inputs: tuple containing the current observation, growth, trend, and seasonality
    """
    lev_prev, slp_prev = carry
    y_t = inputs

    # forecast
    dlt_comp_t = lev_prev + theta * slp_prev

    # update
    new_lev = lev_sm * y_t + (1 - lev_sm) * (lev_prev + theta * slp_prev)
    new_slp = slp_sm * (new_lev - lev_prev) + (1 - slp_sm) * slp_prev

    return (new_lev, new_slp), (new_lev, new_slp, dlt_comp_t)

_, res = lax.scan(lambda carry, inputs: dlt_transition_step(carry, inputs, lev_sm, slp_sm, theta), (y[0], 0), y)
levs, slps, y_hats = res

In [ ]:
lev_sm = 0.00005
slp_sm = 0.0001
theta = 0.9

def dlt_model(lev_sm, slp_sm, theta):
    # variance coefficient
    sigma = numpyro.sample("sigma", dist.HalfNormal(0.5))
    alpha_glb_trend = numpyro.sample("glb_trend_alpha", dist.Normal(0, 1.0))
    beta_glb_trend = numpyro.sample("beta_glb_trend", dist.Normal(0, 1.0))
    beta_seas = numpyro.sample("beta_seas", dist.Normal(0, 0.3).expand([x_seas.shape[1]]))

    # (n_steps, )
    seas = jnp.sum(x_seas * beta_seas, axis=-1)
    # (n_steps, )
    glb_trend = alpha_glb_trend + x_glb_trend * beta_glb_trend
    reg_comp = seas + glb_trend

    # scan with the partial function
    _, res = lax.scan(
        lambda carry, inputs: dlt_transition_step(carry, inputs, lev_sm, slp_sm, theta), 
        (y[0] - reg_comp[0], 0), y - reg_comp
    )
    _, _, dlt_comp = res
    # mid point estimation
    mu = dlt_comp + reg_comp

    numpyro.deterministic("mu", mu)
    numpyro.deterministic("dlt_comp", dlt_comp)

    # likelihood
    numpyro.sample("observations", dist.Normal(loc=mu, scale=sigma), obs=y)

init_strategy = init_to_median(num_samples=10)
kernel = NUTS(dlt_model, init_strategy=init_strategy)
mcmc = MCMC(kernel, num_warmup=1000, num_samples=1000, num_chains=4)
mcmc.run(jax.random.PRNGKey(0), lev_sm=lev_sm, slp_sm=slp_sm, theta=theta)
posteriors_dict = mcmc.get_samples()

In [ ]:
posteriors_dict.keys()

In [ ]:
mu_samples = np.array(posteriors_dict["mu"])
# dlt_mu_samples = np.array(posteriors_dict["dlt_mu"])
sigma_samples = np.array(posteriors_dict["sigma"])

# generate error
eps_samples = np.transpose(
    np.random.normal(loc=0.0, scale=sigma_samples, size=(len(y), sigma_samples.shape[0])),
    axes=(1, 0)
)
print(eps_samples.shape)

# compute p5, p50, p95 for forecast
yhat_lower, yhat_mid, yhat_upper = np.quantile(np.exp(mu_samples + eps_samples) * response_norm, q=[0.05, 0.5, 0.95], axis=0)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(np.arange(len(y)), response, label='Observed', alpha=0.5, color="orange", s=10)
ax.plot(np.arange(len(y)), yhat_mid, label='Forecast', linestyle='--', alpha=0.8, color="dodgerblue")
ax.fill_between(np.arange(len(y)), yhat_lower, yhat_upper, alpha=0.5, label='95% Prediction Interval', color="dodgerblue")
ax.set_title('Damped Local Trend Forecast')

In [ ]:
# how to do a sliding window prediction?
dlt_comp = posteriors_dict["dlt_comp"]
beta_glb_trend = posteriors_dict["beta_glb_trend"]
beta_seas = posteriors_dict["beta_seas"]
sigma = posteriors_dict["sigma"]

reg_comp = np.sum(x_seas * jnp.expand_dims(beta_seas, -2), axis=-1) + x_glb_trend * jnp.expand_dims(beta_glb_trend, -1)
reg_comp.shape

# report shapes
print("dlt_comp shape:", dlt_comp.shape)
print("beta_glb_trend shape:", beta_glb_trend.shape)
print("beta_seas shape:", beta_seas.shape)
print("sigma shape:", sigma.shape)
print("reg_comp shape:", reg_comp.shape)

In [ ]:
import jax
import jax.numpy as jnp

def slice_trend_single(trend, h):
    n_steps = trend.shape[0]
    num_windows = n_steps - h + 1

    # Function to extract one window
    def get_window(t_idx):
        return jax.lax.dynamic_slice(trend, start_indices=(t_idx,), slice_sizes=(h,))

    t_indices = jnp.arange(num_windows)
    return vmap(get_window)(t_indices)  # shape: (num_windows, h)

def slice_trend(trends, h):
    # trends: shape (N_samples, T)
    return vmap(lambda trend: slice_trend_single(trend, h))(trends)

h=28
dlt_comp_slice = dlt_comp[:, :-(h-1), None]
reg_comp_slice = slice_trend(reg_comp, h=h)
yhat_span = dlt_comp_slice + reg_comp_slice
y_slice = slice_trend_single(y, h=h)

print("reg_comp_slice shape:", reg_comp_slice.shape)
print("dlt_comp_slice shape:", dlt_comp_slice.shape)
print("y_slice shape:", y_slice.shape)

In [ ]:
print("yhat_span shape:", yhat_span.shape)
print("sigma shape:", sigma.shape)
print("y_slice shape:", y_slice.shape)

In [ ]:
def compute_log_likelihood(yhat_span, sigma, y_slice):
    # reshape for broadcasting
    sigma_broadcast = sigma[:, None, None]
    y_slice_broadcast = y_slice[None, :, :]

    # compute squared error
    sq_error = (y_slice_broadcast - yhat_span) ** 2

    # compute log likelihood per (s, t, h)
    log_prob = -0.5 * jnp.log(2 * jnp.pi) \
               - jnp.log(sigma_broadcast) \
               - 0.5 * sq_error / (sigma_broadcast ** 2)

    return log_prob

loglk = compute_log_likelihood(yhat_span, sigma, y_slice)
print(loglk.shape)

In [ ]:
def compute_wbic(loglk):
    """
    Args
    ----
    loglk: array-like, log likelihood per samples per obs; should be with values of shape (n_samples, n_steps, h)
    """
    # in original paper, they use sum but it leads to unstable scale due to different size of datasets
    # we use mean instead
    loglk_per_sample = jnp.nanmean(loglk, axis=(-1, -2)) 
    # (n_steps * h, )
    nobs = loglk.shape[-1] * loglk.shape[-2] 
    beta = 1.0 / jnp.log(nobs)  
    wbic = - (1.0 / beta) * jnp.nanmean(loglk_per_sample)
    return wbic

wbic = compute_wbic(loglk)

print("WBIC:", wbic)

Now, let's do a optimization run to find the best hyperparameters for the DLT model.

In [ ]:
h = 28
def run_dlt_model(lev_sm=lev_sm, slp_sm=slp_sm, theta=theta):
    kernel = NUTS(dlt_model, init_strategy=init_to_median(num_samples=10))
    mcmc = MCMC(kernel, num_warmup=1000, num_samples=1000, num_chains=4)
    mcmc.run(jax.random.PRNGKey(0), lev_sm=lev_sm, slp_sm=slp_sm, theta=theta)
    posteriors_dict = mcmc.get_samples()
    
    dlt_comp = posteriors_dict["dlt_comp"]
    beta_glb_trend = posteriors_dict["beta_glb_trend"]
    beta_seas = posteriors_dict["beta_seas"]
    sigma = posteriors_dict["sigma"]

    reg_comp = np.sum(x_seas * jnp.expand_dims(beta_seas, -2), axis=-1) + x_glb_trend * jnp.expand_dims(beta_glb_trend, -1)

    dlt_comp_slice = dlt_comp[:, :-(h-1), None]
    reg_comp_slice = slice_trend(reg_comp, h=h)
    yhat_span = dlt_comp_slice + reg_comp_slice
    y_slice = slice_trend_single(y, h=h)

    loglk = compute_log_likelihood(yhat_span, sigma, y_slice)
    wbic = compute_wbic(loglk)

    # print("WBIC:", wbic)
    return wbic

# test run_dlt_model()
run_dlt_model(lev_sm=lev_sm, slp_sm=slp_sm, theta=theta)